# 🫀 HeartGAPSO Pipeline v3: Comprehensive Heart Disease ML System
### Hybrid Genetic Algorithm (GA) & Particle Swarm Optimization (PSO) with Multi-Model Benchmarking & Deployment Pipeline

This notebook is an end-to-end, self-contained system for heart disease classification on the UCI Cleveland dataset (ID: 45). It includes:
1. **Automated Data Ingestion**: Direct fetching via `ucimlrepo` with automatic local fallback.
2. **Exploratory Data Analysis (EDA)**: Demographic and clinical biomarker distributions.
3. **Statistical Feature Analysis ($T^2$ Scoring)**: Identifies strong discriminative biomarkers for GA seeding.
4. **Baseline Model Benchmarking**: Multi-model comparison (Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM, Random Forest, ExtraTrees, XGBoost, and Stacked Ensemble).
5. **Hybrid GAPSO Optimization Engine**: Joint feature subset selection and Random Forest hyperparameter tuning with multi-seed stability verification.
6. **Convergence Diagnostics & Feature Importance**: Search trajectory inspection and Gini importance ranking.
7. **Threshold Sweeping & Calibration**: Decision threshold tradeoff analysis.
8. **Independent Holdout Evaluation**: Single-pass test evaluation ($N=61$) with Confusion Matrix and ROC-AUC curves.
9. **Artifact Serialization & Real-Time Patient Risk Inference**: Packaging and scoring custom clinical profiles.

---

### 📌 End-to-End Pipeline Architecture
```
┌─────────────────────────────────────────────────────────────┐
│ 1. Data Ingestion: ucimlrepo (id=45) / Local Cleveland Data │
│    Binary target conversion: 0 = Healthy, 1..4 = Disease    │
└──────────────────────────────┬──────────────────────────────┘
                               │
┌──────────────────────────────▼──────────────────────────────┐
│ 2. Stratified 80/20 Train-Test Split (Untouched Holdout)    │
└──────────────────────────────┬──────────────────────────────┘
                               │
┌──────────────────────────────▼──────────────────────────────┐
│ 3. Training-Only Statistical Analysis ($T^2$ Scoring)        │
│    Prioritize key markers: cp, slope, ca, thal              │
└──────────────────────────────┬──────────────────────────────┘
                               │
┌──────────────────────────────▼──────────────────────────────┐
│ 4. Multi-Model Baseline Benchmarking (CV Comparison)        │
│    LR, KNN, NB, DT, SVM, RF, ExtraTrees, XGBoost, Stacking │
└──────────────────────────────┬──────────────────────────────┘
                               │
┌──────────────────────────────▼──────────────────────────────┐
│ 5. Hybrid GA-PSO Optimization Engine                        │
│    - 13-bit chromosome feature subset representation        │
│    - Random Forest hyperparameter optimization              │
│    - Fast screening + Multi-seed stability audit            │
└──────────────────────────────┬──────────────────────────────┘
                               │
┌──────────────────────────────▼──────────────────────────────┐
│ 6. Model Assembly, Threshold Sweeping & Holdout Evaluation  │
│    Accuracy, ROC-AUC, Sensitivity, Specificity, F1          │
└──────────────────────────────┬──────────────────────────────┘
                               │
┌──────────────────────────────▼──────────────────────────────┐
│ 7. Model Serialization & Clinical Patient Risk Inference    │
└─────────────────────────────────────────────────────────────┘
```


## 1. Setup & Environment Configuration
Install and import all necessary data science and machine learning dependencies directly into the active kernel.


In [ ]:
# Install all necessary dependencies into the active notebook kernel
%pip install -q numpy pandas scikit-learn matplotlib ucimlrepo xgboost


In [ ]:
import os
import sys
import json
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from xgboost import XGBClassifier

# Fetch UCI repo
from ucimlrepo import fetch_ucirepo

# Add project root to sys.path
sys.path.insert(0, str(Path.cwd()))

# Import core modular components from src
from src.data_loader import FEATURE_NAMES, load_cleveland
from src.preprocessing import make_preprocessor
from src.statistical_analysis import analyze_features, save_analysis
from src.fitness import FitnessConfig
from src.genetic_algorithm import GAConfig
from src.gapso_optimizer import optimize
from src.random_forest import InferencePipeline, make_random_forest
from src.evaluation import evaluate_model, save_optimization_diagnostics

# Configure plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✓ All libraries imported and environment ready.")


## 2. Dataset Ingestion via `ucimlrepo` (ID: 45)
Fetching the UCI Heart Disease dataset using `fetch_ucirepo(id=45)` with automatic local fallback, mapping targets to binary classification ($0 = \text{Healthy}, 1 = \text{Heart Disease}$).


In [ ]:
try:
    print("Fetching Heart Disease dataset from UCI ML Repository (id=45)...")
    heart_disease = fetch_ucirepo(id=45)
    
    print(f"Dataset Name: {heart_disease.metadata.name}")
    print(f"Total Instances: {heart_disease.metadata.num_instances}, Features: {heart_disease.metadata.num_features}")
    
    # Display variable metadata
    display(heart_disease.variables[['name', 'role', 'type', 'description']].dropna(how='all'))
    
    X_raw = heart_disease.data.features.copy()
    y_raw = heart_disease.data.targets.copy()
    
    X_raw.columns = [str(c).strip() for c in X_raw.columns]
    target_col = y_raw.columns[0]
    
    # Map UCI targets: 0 = Healthy, 1, 2, 3, 4 = Disease Presence (1)
    y_series = pd.to_numeric(y_raw[target_col], errors='coerce')
    y = (y_series > 0).astype(int)
    X = X_raw[FEATURE_NAMES].apply(pd.to_numeric, errors='coerce')
    print(f"✓ Successfully loaded {len(X)} instances from ucimlrepo.")

except Exception as e:
    print(f"Notice: {e}. Falling back to local Cleveland dataset.")
    LOCAL_DATA = Path("heart+disease/processed.cleveland.data")
    X, y = load_cleveland(LOCAL_DATA)
    print(f"✓ Loaded {len(X)} instances from local dataset.")

df_preview = X.copy()
df_preview["target"] = y
display(df_preview.head())


## 3. Exploratory Data Analysis (EDA)
Inspecting class distributions, clinical biomarker statistics, and missing value profiles.


In [ ]:
print(f"Cohort Size: {X.shape[0]} patients, {X.shape[1]} clinical biomarkers")
print(f" - Healthy (Class 0): {(y == 0).sum()} ({(y == 0).mean():.1%})")
print(f" - Heart Disease (Class 1): {(y == 1).sum()} ({(y == 1).mean():.1%})")
print(f"\nMissing values count across features:\n{X.isna().sum()}")

# Summary statistics
display(df_preview.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Target distribution
y.value_counts().plot(kind='bar', ax=axes[0], color=['#2b5c8f', '#d95f02'], edgecolor='k')
axes[0].set_title("Class Balance (0 = Healthy, 1 = Disease)")
axes[0].set_xticklabels(["Healthy (0)", "Disease (1)"], rotation=0)
axes[0].set_ylabel("Patient Count")

# 2. Chest pain vs target
pd.crosstab(X['cp'], y, normalize='index').plot(kind='bar', stacked=True, ax=axes[1], color=['#2b5c8f', '#d95f02'], edgecolor='k')
axes[1].set_title("Chest Pain Type (cp) vs Disease Proportion")
axes[1].set_xlabel("Chest Pain Type (1..4)")
axes[1].set_ylabel("Proportion")
axes[1].legend(["Healthy", "Disease"], loc="upper left")

# 3. Age vs Thalach scatter
scatter = axes[2].scatter(X['age'], X['thalach'], c=y, cmap='coolwarm', alpha=0.85, edgecolors='k')
axes[2].set_title("Age vs. Maximum Heart Rate (thalach)")
axes[2].set_xlabel("Age (years)")
axes[2].set_ylabel("Thalach (bpm)")
cbar = plt.colorbar(scatter, ax=axes[2])
cbar.set_label("Heart Disease Presence")

plt.tight_layout()
plt.show()


## 4. Stratified 80/20 Holdout Split (Untouched Test Partition)
Reserving 20% of the dataset as an untouched holdout partition ($N=61$) for independent evaluation.


In [ ]:
RANDOM_SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED
)

print(f"Training partition (for CV, Feature Selection & GAPSO): {X_train.shape[0]} samples")
print(f"Untouched holdout test partition: {X_test.shape[0]} samples")


## 5. Statistical Feature Significance Analysis ($T^2$ Scoring)
Evaluating feature discriminatory power on the training split to prioritize statistically significant features (`cp`, `slope`, `ca`, `thal`) for Genetic Algorithm seeding.


In [ ]:
statistical_analysis = analyze_features(X_train, y_train)
stats_df = pd.DataFrame(statistical_analysis["features"])

display(stats_df[["importance_rank", "feature", "t2_statistical_score", "class_0_mean", "class_1_mean", "mutation_probability"]])

plt.figure(figsize=(10, 5))
sorted_stats = stats_df.sort_values(by="t2_statistical_score", ascending=True)
bars = plt.barh(sorted_stats["feature"], sorted_stats["t2_statistical_score"], color="#3470a3")
plt.title("Feature Significance via T² Statistical Scoring (Training Cohort)")
plt.xlabel("T² Score")
plt.ylabel("Clinical Feature")

for bar, feat in zip(bars, sorted_stats["feature"]):
    if feat in statistical_analysis["important_features"]:
        bar.set_color("#d95f02")

plt.legend(["Other Features", "Statistically Seeded (cp, slope, ca, thal)"], loc="lower right")
plt.tight_layout()
plt.show()

save_analysis(statistical_analysis, "artifacts/statistical_feature_analysis.json")
print(f"✓ Statistically significant markers identified: {statistical_analysis['important_features']}")


## 6. Baseline Model Benchmarking (Cross-Validation Suite)
Comparing standard supervised classifiers and an ensemble stack on the training cohort using 5-fold Stratified Cross-Validation with fold-local preprocessing (median imputation and MinMax scaling).


In [ ]:
def get_baseline_models(seed=42):
    return {
        "Logistic Regression": LogisticRegression(random_state=seed, solver='liblinear'),
        "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
        "Gaussian Naive Bayes": GaussianNB(),
        "Decision Tree": DecisionTreeClassifier(random_state=seed, max_depth=5),
        "Support Vector Machine": SVC(random_state=seed, probability=True),
        "Random Forest (Default)": RandomForestClassifier(random_state=seed, n_estimators=100),
        "Extra Trees": ExtraTreesClassifier(random_state=seed, n_estimators=100),
        "XGBoost": XGBClassifier(random_state=seed, eval_metric='logloss', use_label_encoder=False),
        "Stacked Ensemble (SVM+XGB+ET)": StackingClassifier(
            estimators=[
                ('svm', SVC(random_state=seed, probability=True)),
                ('xgb', XGBClassifier(random_state=seed, eval_metric='logloss', use_label_encoder=False)),
                ('et', ExtraTreesClassifier(random_state=seed, n_estimators=100))
            ],
            final_estimator=LogisticRegression(random_state=seed),
            cv=3,
            stack_method='predict_proba'
        )
    }

print("Running 5-fold Stratified Cross-Validation on Baseline Models...")
baseline_models = get_baseline_models(RANDOM_SEED)
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

baseline_results = []

for name, clf in baseline_models.items():
    fold_accs, fold_aucs, fold_f1s, fold_sens, fold_specs = [], [], [], [], []
    
    for train_idx, val_idx in cv_splitter.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Fold-local preprocessing to prevent leakage
        prep = make_preprocessor()
        X_tr_p = prep.fit_transform(X_tr)
        X_val_p = prep.transform(X_val)
        
        clf.fit(X_tr_p, y_tr)
        probs = clf.predict_proba(X_val_p)[:, 1]
        preds = (probs >= 0.5).astype(int)
        
        tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
        
        fold_accs.append(accuracy_score(y_val, preds))
        fold_aucs.append(roc_auc_score(y_val, probs))
        fold_f1s.append(f1_score(y_val, preds, zero_division=0))
        fold_sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        fold_specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        
    baseline_results.append({
        "Model": name,
        "CV Accuracy": np.mean(fold_accs),
        "CV ROC-AUC": np.mean(fold_aucs),
        "CV F1-Score": np.mean(fold_f1s),
        "CV Sensitivity": np.mean(fold_sens),
        "CV Specificity": np.mean(fold_specs)
    })

baseline_df = pd.DataFrame(baseline_results).sort_values(by="CV ROC-AUC", ascending=False)
display(baseline_df.style.format({c: "{:.4f}" for c in baseline_df.columns if c != "Model"}))


## 7. Hybrid GAPSO Optimization Engine
Simultaneously optimizing:
- **13-bit Feature Mask**: Selecting the optimal subset of diagnostic biomarkers.
- **Random Forest Hyperparameters**: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, and `class_weight`.

Multi-objective fitness objective:
$$\text{Fitness} = 0.40 \cdot \text{ROC-AUC} + 0.30 \cdot F_1 + 0.15 \cdot \text{Sensitivity} + 0.15 \cdot \text{Specificity} - \lambda_{\text{sparsity}} \cdot |S| - \lambda_{\text{stability}} \cdot \sigma_{\text{CV}}$$


In [ ]:
# Enhanced Fitness Configuration
FITNESS_CONFIG = FitnessConfig(
    folds=5,
    repeats=5,                 # 5-fold CV with 5 repeats for robust evaluation
    roc_auc_weight=0.40,
    f1_weight=0.30,            # Boosted F1 balance
    sensitivity_weight=0.15,   # Balanced sensitivity/specificity
    specificity_weight=0.15,
    feature_penalty=0.002,     # Penalize noisy features
    stability_penalty=0.06     # Penalize cross-fold variance
)

# Deepened GA Configuration
GA_CONFIG = GAConfig(
    population_size=60,
    generations=50,
    pso_iterations=25,
    max_evaluations=1200,
    pso_particles=25
)

print("Starting GAPSO Optimization...")
print(f" - Population: {GA_CONFIG.population_size} | Generations: {GA_CONFIG.generations} | Max Budget: {GA_CONFIG.max_evaluations}")

start_time = time.perf_counter()

opt_result = optimize(
    X=X_train,
    y=y_train,
    seed=RANDOM_SEED,
    ga_config=GA_CONFIG,
    fitness_config=FITNESS_CONFIG,
    high_fidelity_candidate_limit=40,
    stability_candidates=8,
    fast_promotions=40,
    stability_seeds=7,                  # Stability verified across 7 independent seeds
    statistical_analysis=statistical_analysis,
)

opt_runtime = time.perf_counter() - start_time

selected_feature_indices = [idx for idx, bit in enumerate(opt_result["features"]) if bit]
selected_feature_names = [FEATURE_NAMES[idx] for idx in selected_feature_indices]

print(f"\n✓ GAPSO Optimization completed in {opt_runtime:.2f} seconds!")
print(f"Best Fitness Score: {opt_result['fitness']:.4f}")
print(f"Selected Features ({len(selected_feature_names)}/13): {selected_feature_names}")
print(f"Optimal Hyperparameters: {json.dumps(opt_result['params'], indent=2)}")
print(f"Calibrated Decision Threshold: {opt_result['threshold']:.3f}")


## 8. Optimization Diagnostics & Search Trajectories


In [ ]:
history = opt_result.get("history", [])

if history:
    generations = [item["generation"] for item in history]
    best_fitness_curve = [item["best_fitness"] for item in history]
    mean_fitness_curve = [item["mean_fitness"] for item in history]
    mean_features_curve = [np.mean(item["selected_feature_counts"]) for item in history]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Fitness Convergence
    axes[0].plot(generations, best_fitness_curve, 'o-', color='#1b9e77', label='Best Fitness', linewidth=2)
    axes[0].plot(generations, mean_fitness_curve, 's--', color='#7570b3', label='Population Mean Fitness', alpha=0.8)
    axes[0].set_title("GAPSO Fitness Convergence across Generations")
    axes[0].set_xlabel("Generation")
    axes[0].set_ylabel("Fitness Score")
    axes[0].legend()

    # Feature Sparsity Trajectory
    axes[1].plot(generations, mean_features_curve, '^-', color='#d95f02', linewidth=2)
    axes[1].set_title("Mean Selected Features across Generations")
    axes[1].set_xlabel("Generation")
    axes[1].set_ylabel("Selected Features")

    plt.tight_layout()
    plt.show()

# Display Top Candidate Audit
candidate_audit = pd.DataFrame(opt_result["candidate_audit"])
if not candidate_audit.empty:
    cols = ["candidate_id", "fast_rank", "screening_rank", "final_rank", "screening_fitness", "final_fitness", "selected_feature_count"]
    display(candidate_audit[[c for c in cols if c in candidate_audit.columns]].head(10))


## 9. Final Model Assembly & Gini Feature Importance
Fitting the optimized Random Forest classifier with fold-local preprocessing on the selected feature subset, and packaging into a self-contained `InferencePipeline`.


In [ ]:
# 1. Fit Preprocessor (Median Imputation & MinMax Scaling) on selected features
preprocessor = make_preprocessor()
X_train_selected = X_train.iloc[:, selected_feature_indices]
X_train_processed = preprocessor.fit_transform(X_train_selected)

# 2. Train Random Forest with optimal hyperparameters
final_rf = make_random_forest(opt_result["params"], RANDOM_SEED)
final_rf.fit(X_train_processed, y_train)

# 3. Assemble Inference Pipeline
pipeline = InferencePipeline(
    preprocessing=preprocessor,
    classifier=final_rf,
    indices=selected_feature_indices,
    threshold=opt_result["threshold"]
)

# Plot Gini Feature Importance
importances = final_rf.feature_importances_
feat_imp = pd.Series(importances, index=selected_feature_names).sort_values(ascending=True)

plt.figure(figsize=(9, 4.5))
feat_imp.plot(kind='barh', color='#2b5c8f', edgecolor='k')
plt.title("Random Forest Gini Feature Importance (Selected Biomarkers)")
plt.xlabel("Importance Weight")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("✓ Final Inference Pipeline trained and assembled.")


## 10. Decision Threshold Sweeping & Tradeoff Analysis
Visualizing the performance tradeoffs across decision thresholds ($0.30 - 0.70$) on the untouched holdout partition.


In [ ]:
y_test_prob = pipeline.predict_proba(X_test)[:, 1]

threshold_sweep = []
test_thresholds = np.linspace(0.30, 0.70, 41)

for t in test_thresholds:
    preds = (y_test_prob >= t).astype(int)
    tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
    threshold_sweep.append({
        "Threshold": t,
        "Accuracy": accuracy_score(y_test, preds),
        "Sensitivity": recall_score(y_test, preds, zero_division=0),
        "Specificity": tn_s / (tn_s + fp_s) if (tn_s + fp_s) > 0 else 0,
        "Precision": precision_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0)
    })

sweep_df = pd.DataFrame(threshold_sweep)

plt.figure(figsize=(11, 5))
plt.plot(sweep_df["Threshold"], sweep_df["Accuracy"], 'o-', label='Accuracy', lw=2)
plt.plot(sweep_df["Threshold"], sweep_df["Precision"], 's--', label='Precision', lw=2)
plt.plot(sweep_df["Threshold"], sweep_df["Sensitivity"], '^-', label='Sensitivity (Recall)', lw=2)
plt.plot(sweep_df["Threshold"], sweep_df["Specificity"], 'd-', label='Specificity', lw=2)
plt.axvline(pipeline.threshold, color='red', linestyle=':', label=f'Optimized Threshold ({pipeline.threshold:.2f})')

plt.title("Holdout Metric Tradeoffs across Classification Thresholds")
plt.xlabel("Classification Threshold")
plt.ylabel("Metric Score")
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()


## 11. Independent Holdout Evaluation (Test Partition, N=61)
Scoring the untouched holdout test partition with the final pipeline.


In [ ]:
y_test_pred = pipeline.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred, labels=[0, 1]).ravel()
acc = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred, zero_division=0)
rec = recall_score(y_test, y_test_pred, zero_division=0)
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
f1 = f1_score(y_test, y_test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_test_prob)

metrics_summary = {
    "Accuracy": acc,
    "ROC-AUC": roc_auc,
    "Sensitivity / Recall": rec,
    "Specificity": spec,
    "Precision": prec,
    "F1-Score": f1,
    "Classification Threshold": pipeline.threshold
}

metrics_df = pd.DataFrame(list(metrics_summary.items()), columns=["Metric", "Holdout Value"])
display(metrics_df.style.format({"Holdout Value": "{:.4f}"}))

# Plots: Confusion Matrix and ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=np.array([[tn, fp], [fn, tp]]),
    display_labels=["Healthy (0)", "Disease (1)"]
)
cm_display.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Holdout Confusion Matrix (N={len(y_test)}, Acc={acc:.1%})")
axes[0].grid(False)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
axes[1].plot(fpr, tpr, color='#1b9e77', lw=2.5, label=f'HeartGAPSO Model (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Chance (AUC = 0.500)')
axes[1].set_title("Receiver Operating Characteristic (ROC)")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)")
axes[1].set_ylabel("True Positive Rate (Sensitivity)")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


## 12. Model Serialization & Artifact Export
Persisting trained pipeline binaries and JSON metric reports to `models/`, `results/`, and `artifacts/`.


In [ ]:
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

report = {
    **metrics_summary,
    "selected_features": selected_feature_names,
    "number_selected_features": len(selected_feature_names),
    "best_rf_parameters": opt_result["params"],
    "optimization_fitness": opt_result["fitness"],
    "optimization_runtime_seconds": opt_runtime,
    "classification_threshold": pipeline.threshold,
    "seed": RANDOM_SEED,
    "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]]
}

# Save serialized models
with open("models/gapso_random_forest.pkl", "wb") as f:
    pickle.dump(pipeline, f)
with open("models/preprocessing.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

Path("models/selected_features.json").write_text(json.dumps(selected_feature_names, indent=2))
Path("models/model_config.json").write_text(json.dumps(report, indent=2, default=str))
Path("results/evaluation.json").write_text(json.dumps(report, indent=2, default=str))
Path("results/metrics.json").write_text(json.dumps(report, indent=2, default=str))

save_optimization_diagnostics(opt_result["history"], opt_result, "results")
print("✓ Serialized model pipeline and evaluation reports saved to models/, results/, artifacts/.")


## 13. Interactive Clinical Patient Risk Inference
Demonstrating real-time probability scoring and risk classification on new patient clinical profiles.


In [ ]:
sample_cohort = pd.DataFrame([
    {
        "Patient": "Patient A (Severe Presentation)",
        "age": 67, "sex": 1, "cp": 4, "trestbps": 160, "chol": 286, "fbs": 1,
        "restecg": 2, "thalach": 108, "exang": 1, "oldpeak": 2.6, "slope": 2,
        "ca": 3, "thal": 7
    },
    {
        "Patient": "Patient B (Moderate Presentation)",
        "age": 58, "sex": 1, "cp": 3, "trestbps": 140, "chol": 211, "fbs": 0,
        "restecg": 0, "thalach": 165, "exang": 0, "oldpeak": 0.8, "slope": 1,
        "ca": 0, "thal": 3
    },
    {
        "Patient": "Patient C (Low Risk Healthy)",
        "age": 41, "sex": 0, "cp": 2, "trestbps": 120, "chol": 175, "fbs": 0,
        "restecg": 0, "thalach": 178, "exang": 0, "oldpeak": 0.0, "slope": 1,
        "ca": 0, "thal": 3
    }
])

patient_features = sample_cohort[FEATURE_NAMES]
probabilities = pipeline.predict_proba(patient_features)[:, 1]
predictions = pipeline.predict(patient_features)

demo_results = []
for i, row in sample_cohort.iterrows():
    prob = probabilities[i]
    pred = predictions[i]
    demo_results.append({
        "Patient": row["Patient"],
        "Age": row["age"],
        "Sex": "Male" if row["sex"] == 1 else "Female",
        "Chest Pain Type": row["cp"],
        "Max HR (thalach)": row["thalach"],
        "Predicted Risk Prob": f"{prob:.1%}",
        "Decision Threshold": f"{pipeline.threshold:.3f}",
        "Risk Classification": "HIGH RISK (Disease)" if pred == 1 else "LOW RISK (Healthy)"
    })

display(pd.DataFrame(demo_results))


---
### ✅ HeartGAPSO Pipeline Execution Complete
The trained model is stored at `models/gapso_random_forest.pkl` and ready for clinical evaluation.
